# Load Data

In [10]:
TABLE = "price_features"
CALENDAR = "no_calendar"

In [11]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\maxan\OneDrive\Desktop\0. Personal Projects\market-intelligence-pipeline")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [12]:
import pandas as pd
import numpy as np
import duckdb

from src.config import (
    DATABASE_PATH,
    ROLLING_WINDOWS,
    LAGGED_WINDOWS,
    MA_WINDOWS,
    TICKERS,
    START_DATE,
    END_DATE
)

from src.features import (
    build_price_feature_columns
)

CALENDAR = "no_calendar"


def pull_calendar_table(table_name: str, calendar: str) -> pd.DataFrame:

    print("[START] Connecting to database...")

    con = duckdb.connect(DATABASE_PATH)

    print(f"[START] Reading {table_name}")

    df = con.sql(f"""
    
        SELECT *
        FROM {table_name}

    """).df()

    print(f"[DONE] Read {table_name}")

    con.close()

    print(f"[START] Building {CALENDAR} required columns...")

    base_cols = [
        "ticker",
        "date",
        "open",
        "high",
        "low",
        "close",
        "adj_close",
        "volume"
    ]

    feature_columns = build_price_feature_columns(
        calendars = [CALENDAR],
        rolling_windows = ROLLING_WINDOWS,
        lagged_windows = LAGGED_WINDOWS,
        ma_windows = MA_WINDOWS
    )

    expected_columns = base_cols + list(feature_columns.keys())

    calendar_df = df[expected_columns].copy()

    print(f"[DONE] Filtered to {CALENDAR}")

    calendar_df["date"] = pd.to_datetime(calendar_df["date"])

    calendar_df = calendar_df.sort_values(["ticker", "date"])

    print(f"[DONE] Prepared and sorted table")

    return calendar_df

In [13]:
df = pull_calendar_table(
    table_name = TABLE,
    calendar = CALENDAR
)

[START] Connecting to database...
[START] Reading price_features
[DONE] Read price_features
[START] Building no_calendar required columns...
[DONE] Filtered to no_calendar
[DONE] Prepared and sorted table


In [61]:
table_count = 0
graph_count = 0

# Missingness by column

In [62]:
df_1 = df.copy()

missing_table = (
    df_1.isna().sum().reset_index().rename(columns={
        "index": "Column",
        0: "Missing Values"
    })
)

table_count +=1
print(f"Table {table_count}:")

display(missing_table.sort_values("Missing Values", ascending=False).style.hide(axis="index"))

Table 1:


Column,Missing Values
moving_avg_50_no_calendar,343
rolling_30d_return_no_calendar,210
rolling_30d_volatility_no_calendar,210
price_vs_ma20_no_calendar,133
moving_avg_20_no_calendar,133
relative_volume_no_calendar,133
rolling_7d_return_no_calendar,49
lag_5_return_no_calendar,42
lag_1_return_no_calendar,14
cumulative_returns_no_calendar,7


In [63]:
ticker = "ticker"
date = "date"

missing_cols = df_1.columns[df_1.isna().any()].tolist()

def total_missing(x):
    return int(x.isna().sum())

# Takes a series as an input, 
# starting at the start and moving forwards, 
# creating a cumulative count of missing values until
# the first non missing value appears
def leading_missing(x):
    return int(x.isna().cumprod().sum())

# Takes a series as an input, 
# starting at the end and moving backward, 
# creating a cumulative count of missing values until 
# the first non missing value appears.
def trailing_missing(x):
    return int(x.iloc[::-1].isna().cumprod().sum())

audit_rows = []

for col in missing_cols:
    per_ticker = (
        df_1.groupby(ticker)[col]
        .agg(
            total_missing=total_missing,
            leading_missing=leading_missing,
            trailing_missing=trailing_missing
        )
    )

    per_ticker["internal_missing"] = (
        per_ticker["total_missing"]
        - per_ticker["leading_missing"]
        - per_ticker["trailing_missing"]
    )

    audit_rows.append({
        "Column": col,
        "Total Missing": per_ticker["total_missing"].sum(),
        "Min Missing per Ticker": per_ticker["total_missing"].min(),
        "Max Missing per Ticker": per_ticker["total_missing"].max(),
        "Leading Missing Range": f"{per_ticker['leading_missing'].min()} - {per_ticker['leading_missing'].max()}",
        "Trailing Missing Range": f"{per_ticker['trailing_missing'].min()} - {per_ticker['trailing_missing'].max()}",
        "Internal Missing": per_ticker["internal_missing"].sum()
    })

missingness_audit = pd.DataFrame(audit_rows)

table_count +=1
print(f"Table {table_count}:")

display(missingness_audit.style.hide(axis="index"))


Table 2:


Column,Total Missing,Min Missing per Ticker,Max Missing per Ticker,Leading Missing Range,Trailing Missing Range,Internal Missing
daily_return_no_calendar,7,1,1,1 - 1,0 - 0,0
log_return_no_calendar,7,1,1,1 - 1,0 - 0,0
cumulative_returns_no_calendar,7,1,1,1 - 1,0 - 0,0
rolling_7d_return_no_calendar,49,7,7,7 - 7,0 - 0,0
rolling_30d_return_no_calendar,210,30,30,30 - 30,0 - 0,0
lag_1_return_no_calendar,14,2,2,2 - 2,0 - 0,0
lag_5_return_no_calendar,42,6,6,6 - 6,0 - 0,0
moving_avg_20_no_calendar,133,19,19,19 - 19,0 - 0,0
moving_avg_50_no_calendar,343,49,49,49 - 49,0 - 0,0
price_vs_ma20_no_calendar,133,19,19,19 - 19,0 - 0,0


# Missingness by ticker

In [64]:
df_2 = df.copy()

missing_table = (
    df_2.groupby("ticker").apply(lambda x: x.isna().sum().sum())
        .reset_index()
        .rename(columns={
            "index": "Ticker",
            0: "Missing Values"
        })
)

table_count +=1
print(f"Table {table_count}:")

display(missing_table.sort_values("Missing Values", ascending=False).style.hide(axis="index"))

Table 3:


ticker,Missing Values
GLD,186
MU,186
NKE,186
RPI.L,186
SNDK,186
SPY,186
TLT,186


# Duplicates

In [24]:
df_3 = df.copy()

print("Number of duplicate ticker, date rows: ", df_3.duplicated(subset=["ticker", "date"]).sum())

Number of duplicate ticker, date rows:  0


# Infinite values

In [65]:
df_4 = df.copy()
number_df = df_4.select_dtypes(include="number")

inf_table = (
    np.isinf(number_df).sum()
        .reset_index()
        .rename(columns = {
            "index": "Column",
            0: "Inf Values"
        })
)

table_count +=1
print(f"Table {table_count}:")

display(inf_table.style.hide(axis="index"))

Table 4:


Column,Inf Values
open,0
high,0
low,0
close,0
adj_close,0
volume,0
daily_return_no_calendar,0
log_return_no_calendar,0
cumulative_returns_no_calendar,0
rolling_7d_return_no_calendar,0


# Zero or negative prices

In [66]:
df_5 = df.copy()
price_cols = ["open", "high", "low", "close", "adj_close"]
price_df = df[price_cols]

invalid_table = (
    (price_df <= 0).sum()
        .reset_index()
        .rename(columns = {
            "index": "Column",
            0 : "Invalid Values"
        })
)

table_count +=1
print(f"Table {table_count}:")

display(invalid_table.style.hide(axis="index"))

Table 5:


Column,Invalid Values
open,0
high,0
low,0
close,0
adj_close,0


# Date gaps per ticker

In [67]:
df_6 = df.copy()

ticker = "ticker"
date = "date"

df_6["previous_date"] = df_6.groupby(ticker)[date].shift()
df_6["gap"] = df_6[date] - df_6["previous_date"]

gap_table = (
    df_6.assign(Date_Gap=df_6["gap"] > pd.Timedelta(days=1))
        .groupby(ticker, as_index=False)["Date_Gap"]
        .sum()
        .rename(columns={
            ticker: "Ticker",
            "Date_Gap": "Date Gaps"
        })
)

table_count +=1
print(f"Table {table_count}:")

display(gap_table.style.hide(axis="index"))

Table 6:


Ticker,Date Gaps
GLD,467
MU,467
NKE,467
RPI.L,110
SNDK,76
SPY,467
TLT,467


# Enough data

In [68]:
df_7 = df.copy()

windows = ROLLING_WINDOWS + LAGGED_WINDOWS + MA_WINDOWS
max_window = max(windows)

data_df = df_7.groupby(ticker).size()

data_table = (
    df_7.groupby(ticker, as_index=False)
        .agg(
            first_date=(date, "min"),
            last_date=(date, "max"),
            observations=(date, "size")
        )
)

data_table["sufficient_data"] = (
    data_table["observations"] > (2 * max_window)
)

data_table = data_table.sort_values("observations")

table_count +=1
print(f"Table {table_count}:")

display(data_table.style.hide(axis="index"))


Table 7:


ticker,first_date,last_date,observations,sufficient_data
SNDK,2025-02-13 00:00:00,2026-06-30 00:00:00,345,True
RPI.L,2024-06-11 00:00:00,2026-06-30 00:00:00,519,True
MU,2018-01-02 00:00:00,2026-06-30 00:00:00,2134,True
GLD,2018-01-02 00:00:00,2026-06-30 00:00:00,2134,True
NKE,2018-01-02 00:00:00,2026-06-30 00:00:00,2134,True
SPY,2018-01-02 00:00:00,2026-06-30 00:00:00,2134,True
TLT,2018-01-02 00:00:00,2026-06-30 00:00:00,2134,True


# Expected nulls

In [69]:
df_8 = df.copy()

target_df = df_8.loc[:, df_8.columns.str.contains("target|ticker", case=False)]

target_table = target_df.groupby("ticker").tail(1)

table_count +=1
print(f"Table {table_count}:")

display(target_table.style.hide(axis="index"))

Table 8:


ticker,target_next_day_return_no_calendar,target_direction_no_calendar
GLD,nan,
MU,nan,
NKE,nan,
RPI.L,nan,
SNDK,nan,
SPY,nan,
TLT,nan,
